In [40]:
import sys
import os
import importlib
import pandas as pd
import numpy as np

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from scripts.data_loading import consumers, accounts, transactions, category_mapping
from scripts.backfill_transactions import build_backfill_df
import scripts.feature_creation as _fc_module
importlib.reload(_fc_module)
from scripts.feature_creation import create_all_features, print_feature_summary, print_feature_groups

# check if feature_creation_backfill_df.csv exists, and load it instead of rebuilding 
if os.path.exists('../output/feature_creation_backfill_df.csv'):
    print("✓ Loaded existing 'feature_creation_backfill_df.csv'")
    df = pd.read_csv('../output/feature_creation_backfill_df.csv')
    df["prism_consumer_id"] = pd.to_numeric(df["prism_consumer_id"], errors="coerce").astype("Int64")
else:
    df = build_backfill_df()
    
df.head()

✓ Loaded existing 'feature_creation_backfill_df.csv'


,prism_consumer_id,date,balance,credit_or_debit,amount_change,DQ_TARGET
0,3023,2021-08-31,225.95,starting value,0.00,0.0
1,3023,2021-03-24,205.32,DEBIT,20.63,0.0
2,3023,2021-03-27,445.32,CREDIT,240.00,0.0
3,3023,2021-03-27,785.32,CREDIT,340.00,0.0
4,3023,2021-03-29,760.32,DEBIT,25.00,0.0


In [41]:
df['prism_consumer_id'].nunique()

12900

## Feature Creation

All feature creation logic has been moved to `scripts/feature_creation.py`. Features are organised into four behavioural dimensions plus a multi-account layer:

---

### 1. Balance Behaviour
Features describing the checking account balance level, shape, and risk over time.

| Group | Window | # Features | Description |
|---|---|---|---|
| Balance stats | All-time | 10 | mean, median, min, max, std, % negative, % below $100, % below $500, # days, account history length |
| Balance stats | 30d / 60d / 90d / 180d | 4 × 4 = 16 | mean, min, std, % negative per window |
| Low balance risk | All-time, 30d, 90d | 5 × 3 = 15 | days below $0/$50/$100, max consecutive negative days, zero-crossing count |
| Paycheck-to-paycheck | 90d | 3 | avg min balance before income, depletion rate, days to deplete half balance |

**Balance behaviour subtotal: 44 features**

---

### 2. Cashflow Behaviour
Features describing the flow of money into and out of the checking account.

| Group | Window | # Features | Description |
|---|---|---|---|
| Daily cashflow | 30d / 60d / 90d / 180d | 5 × 4 = 20 | net cashflow, mean daily cashflow, volatility, # transactions, # active days per window |
| Income regularity | 90d | 7 | income frequency, coefficient of variation, avg days between deposits, income count; paycheck regularity flag, consistency score, paycheck count |

**Cashflow behaviour subtotal: 27 features**

---

### 3. Transaction Activity
Features describing the volume, size, and composition of raw transactions.

| Group | Window | # Features | Description |
|---|---|---|---|
| Transaction stats | All-time | 7 | total count, std amount, total credit, total debit, max single credit, max single debit, credit-to-debit ratio |
| Overdraft & fees | All-time, 30d, 90d | 15 | overdraft fee count/total, account fee count/total per window; aggregated total fees per window |

**Transaction activity subtotal: 22 features**

---

### 4. Category-Level Spending
Features describing spend patterns across merchant categories.

| Group | Window | # Features | Description |
|---|---|---|---|
| Per-category features | All-time + 90d | 120 | net spend + transaction count for each of the top 30 categories (30 cats × 2 stats × 2 windows) |
| Grouped categories | All-time | 5 | income total, essentials spend, discretionary spend, essentials-to-income ratio, discretionary-to-income ratio |

**Category-level subtotal: 125 features**

---

### 5. Multi-Account Features
Features derived from non-checking accounts in the accounts table (latest balance snapshot per account).

| Group | Account types | # Features | Description |
|---|---|---|---|
| Liquid / Savings | `SAVINGS`, `MONEYMARKET`, `MONEY MARKET`, `CASH MANAGEMENT`, `PREPAID` | 5 | latest/mean/min/max balance, # accounts |
| Investment / Retirement | `ROTH`, `RETIREMENT`, `BROKERAGE`, `IRA`, `401K`, `STOCK PLAN`, `HSA`, `CD` | 3 | total/mean balance, # accounts |
| Credit Card | `CREDIT CARD` | 4 | latest/mean/max debt, # accounts |
| Revolving credit | `LINE OF CREDIT`, `OVERDRAFT` | 3 | latest/mean balance, # accounts |
| Long-term debt | `LOAN`, `MORTGAGE`, `AUTO`, `STUDENT`, `HOME EQUITY`, `CONSUMER` | 3 | total/mean debt, # accounts |
| Consumer-level aggregates | All types | 19 | liquid/investment/revolving/long-term totals + means; total assets, total debt, net worth, account-type diversity, total account count |
| Composites | Derived | 3 | `net_liquid_position__all_accounts`, `total_debt__all_accounts`, `savings_to_debt_ratio` |

**Multi-account subtotal: 40 features**


In [42]:
# Create all features using the feature_creation module
# Pass accounts to include SAVINGS and CREDIT CARD features
features_df = create_all_features(df, transactions, category_mapping, accounts_df=accounts, consumers_df=consumers)

# Display first few rows
features_df.head()

FEATURE CREATION PIPELINE
Preparing daily data...
Daily data shape: (1183217, 6)
Creating balance features...
Balance features shape: (12900, 10)
Creating daily window features...
  30d: (12900, 9)
  60d: (12900, 9)
  90d: (12900, 9)
  180d: (12900, 9)
Creating transaction features...
Transaction features shape: (12900, 7)
Creating category features (top 30 categories)...
Category features (all-time): (14491, 60)
Category features (90d): (14491, 60)
Creating grouped category features...
Grouped category features shape: (14305, 5)
Creating overdraft & fee features...
Overdraft & fee features shape: (7219, 15)
Creating low balance risk features...
Low balance risk features shape: (12900, 15)
Creating income regularity features...
Income regularity features shape: (14035, 7)
Creating paycheck-to-paycheck features...
Paycheck-to-paycheck features shape: (12900, 3)
Creating consumer-level balance features (all account types)...
Consumer-level balance features shape: (13009, 19)
Creating mul

,balance__mean__all,balance__median__all,balance__min__all,balance__max__all,balance__std__all,balance__pct_negative__all,balance__pct_below_100__all,balance__pct_below_500__all,n_days__all,account__history_days__all,...,revolving_debt__balance__latest,revolving_debt__balance__mean,revolving_debt__n_accounts,longterm_debt__balance__total,longterm_debt__balance__mean,longterm_debt__n_accounts,net_liquid_position__all_accounts,total_debt__all_accounts,savings_to_debt_ratio,DQ_TARGET
prism_consumer_id,,,,,,,,,,,,,,,,,,,,,
0,276.961538,70.090,-1019.10,2732.86,1016.288836,0.475524,0.510490,0.580420,143.0,179.0,...,0.0,0.0,0.0,0.0,0.0,0.0,25.70,0.0,2.570000e+10,0.0
1,1674.533585,1758.350,-123.25,3597.09,1159.524803,0.056604,0.094340,0.301887,106.0,180.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3211.18,0.0,3.211180e+12,0.0
2,2184.981268,890.205,-644.93,6749.98,2451.867841,0.098592,0.133803,0.267606,142.0,180.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2561.43,0.0,2.561430e+12,0.0
3,258.308151,461.340,-2790.64,4229.79,1854.676304,0.445378,0.445378,0.512605,119.0,180.0,...,0.0,0.0,0.0,0.0,0.0,0.0,6690.19,0.0,6.690190e+12,0.0
4,-786.847670,-725.410,-2151.98,753.51,634.556877,0.902913,0.922330,0.990291,103.0,438.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.93,0.0,2.930000e+09,0.0


In [43]:
features_df.index.nunique()

15000

In [45]:
# Save features for later use
features_df.to_csv('../output/new_features.csv', index=True)

## Feature Analysis

In [46]:
# Print feature summary statistics
summary = print_feature_summary(features_df)

# Also return the summary dataframe for further analysis
summary.head(30)


Top 30 features by non-zero rate:
                                 feature  nonzero_rate          mean           std
36                           n_days__90d      1.000000     52.100465     22.737090
46                            tx__n__all      1.000000    403.974806    367.633269
27                           n_days__60d      1.000000     35.677752     15.203484
35                             n_tx__90d      1.000000    241.500775    197.009219
18                           n_days__30d      1.000000     18.649147      7.663812
17                             n_tx__30d      1.000000     86.451628     71.136098
44                            n_tx__180d      1.000000    358.417674    294.484663
45                          n_days__180d      1.000000     82.253721     46.824802
26                             n_tx__60d      1.000000    165.902868    135.837156
8                            n_days__all      1.000000     91.722248     56.219880
233   consumer_balance__n_total_accounts      1.0000

,feature,nonzero_rate,mean,std
36,n_days__90d,1.000000,52.100465,22.737090
46,tx__n__all,1.000000,403.974806,367.633269
27,n_days__60d,1.000000,35.677752,15.203484
35,n_tx__90d,1.000000,241.500775,197.009219
18,n_days__30d,1.000000,18.649147,7.663812
17,n_tx__30d,1.000000,86.451628,71.136098
44,n_tx__180d,1.000000,358.417674,294.484663
45,n_days__180d,1.000000,82.253721,46.824802
26,n_tx__60d,1.000000,165.902868,135.837156
8,n_days__all,1.000000,91.722248,56.219880


In [47]:
# Print features organized by groups
print_feature_groups(features_df)


FEATURE GROUPS

Balance behavior (40 features):
  - balance__mean__all
  - balance__median__all
  - balance__min__all
  - balance__max__all
  - balance__std__all
  - balance__pct_negative__all
  - balance__pct_below_100__all
  - balance__pct_below_500__all
  - balance__mean__30d
  - balance__min__30d
  - balance__std__30d
  - balance__pct_negative__30d
  - balance__mean__60d
  - balance__min__60d
  - balance__std__60d
  - balance__pct_negative__60d
  - balance__mean__90d
  - balance__min__90d
  - balance__std__90d
  - balance__pct_negative__90d
  - balance__mean__180d
  - balance__min__180d
  - balance__std__180d
  - balance__pct_negative__180d
  - balance__days_below_zero__all
  - balance__days_below_50__all
  - balance__days_below_100__all
  - balance__consecutive_negative_days_max__all
  - balance__zero_crossings_count__all
  - balance__days_below_zero__30d
  - balance__days_below_50__30d
  - balance__days_below_100__30d
  - balance__consecutive_negative_days_max__30d
  - balance__

## Model Evaluation

In [48]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Prepare data
eval_df = features_df.replace([np.inf, -np.inf], np.nan)
eval_df = eval_df[eval_df["DQ_TARGET"].notna()].copy()

y = eval_df["DQ_TARGET"].astype(int)
X = (
    eval_df.drop(columns=["DQ_TARGET"])
      .select_dtypes(include=[np.number])
      .fillna(0)
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Tree-based gradient boosting (handles nonlinearity)
model = HistGradientBoostingClassifier(
    max_depth=6,
    max_iter=300,
    learning_rate=0.05,
    min_samples_leaf=50,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluate
val_prob = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_prob)

print("HistGradientBoosting AUC:", round(auc, 4))

HistGradientBoosting AUC: 0.7831
